# cells.jsonl 방출기 테스트 (작업지시서 §4.4)

`emit_cells.py`가 만든 원장(cells.jsonl)을 눈으로 확인하는 노트북입니다.
**한 줄 = 한 셀**이고, 각 줄에는 셀 주소·값·수식·서식이 고정된 순서로 들어갑니다.

확인하는 것:
1. 한 파일을 cells.jsonl로 만들어 내용 보기
2. 골든 셀 — 지시서 §4.4 예시 JSON과 완전 일치하는지
3. 포함 규칙 — 무엇이 살아남고 무엇이 빠지는지
4. used range 재계산(D-01) — 파일 기록보다 실제 내용이 이긴다
5. 결정론(P2) — 두 번 만들어도 바이트까지 같은지
6. 32종 전수 방출

> 실행 전 커널이 이 프로젝트의 `.venv`인지 확인하세요.

In [1]:
# 0. 준비 — 경로 자동 판별과 import
from pathlib import Path
import hashlib, json, sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():   # tests/ 안에서 열었을 때
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from excel_to_skill.extractor import extract_workbook
from excel_to_skill.emit_cells import iter_cell_records, write_cells_jsonl

RAW = ROOT / "workingpaper_raw"   # 실조서 코퍼스 (git 밖)
OUT = ROOT / "tests" / "_output"  # 노트북 산출물 (git 밖)
OUT.mkdir(exist_ok=True)
print("프로젝트 루트:", ROOT)
print("코퍼스 파일 수:", len(list(RAW.glob("*.xls*"))))

프로젝트 루트: /home/shin/Project/Excel_to_Skill
코퍼스 파일 수: 32


## 1. 한 파일을 cells.jsonl로 만들어 보기

감사계약 파일을 추출(extractor) → 방출(emit_cells)해서 실제 파일을 만들고,
앞 5줄을 그대로 읽어봅니다. `sheet`(시트명)·`cell`(주소)·`value`(값)·`formula`(수식)
순서가 모든 줄에서 똑같습니다 — 이 고정 순서가 결정론(P2)의 일부입니다.

In [2]:
gyeyak = next(RAW.glob("*1100~1300*"))
ir = extract_workbook(gyeyak)
jsonl_path = OUT / "감사계약.cells.jsonl"
n = write_cells_jsonl(ir, jsonl_path)
print(f"원본: {gyeyak.name}")
print(f"방출: {jsonl_path.name} — {n}줄\n")

for line in jsonl_path.read_text(encoding="utf-8").splitlines()[:5]:
    print(line)

원본: 감사조서서식_1100~1300 감사계약.xlsx
방출: 감사계약.cells.jsonl — 208줄

{"sheet":"1100","cell":"A2","row":2,"col":1,"value":"1100 계약전 위험평가 (기준서 210)","formula":null,"cached_value":null,"data_type":"s","number_format":"General","merged_range":"A2:F2","bold":true,"border":false,"fill":null}
{"sheet":"1100","cell":"A4","row":4,"col":1,"value":"회사명","formula":null,"cached_value":null,"data_type":"s","number_format":"General","merged_range":null,"bold":false,"border":false,"fill":null}
{"sheet":"1100","cell":"B4","row":4,"col":2,"value":null,"formula":null,"cached_value":null,"data_type":"n","number_format":"General","merged_range":null,"bold":false,"border":true,"fill":"#FFFFCC"}
{"sheet":"1100","cell":"C4","row":4,"col":3,"value":"작성자","formula":null,"cached_value":null,"data_type":"s","number_format":"General","merged_range":null,"bold":false,"border":false,"fill":null}
{"sheet":"1100","cell":"D4","row":4,"col":4,"value":null,"formula":null,"cached_value":null,"data_type":"n","number_format":"yyyy\"

## 2. 골든 셀 — 지시서 예시와 비교

지시서 §4.4에는 `1200!B4`가 어떻게 나와야 하는지 예시 JSON이 그대로 실려 있습니다.
우리 방출 결과가 **필드 하나하나까지 같은지** 비교합니다.

또 `1200!B5`는 지시서가 직접 언급한 실측 케이스입니다 — 빈 날짜 셀을 참조하는
수식이라 계산값이 파이썬 `time(0, 0)` 객체로 나오는데, 이를 JSON에 넣을 때
ISO 8601 문자열 `"00:00:00"`으로 고정 직렬화해야 합니다(§4.4 직렬화 규칙).

In [3]:
recs = {(r["sheet"], r["cell"]): r for r in iter_cell_records(ir)}

spec_b4 = {"sheet": "1200", "cell": "B4", "row": 4, "col": 2, "value": None,
           "formula": "'1100'!B4", "cached_value": 0, "data_type": "f",
           "number_format": "General", "merged_range": None, "bold": False,
           "border": True, "fill": None}
b4 = recs[("1200", "B4")]
print("우리 방출  :", json.dumps(b4, ensure_ascii=False))
print("지시서 예시:", json.dumps(spec_b4, ensure_ascii=False))
diffs = {k: (b4[k], v) for k, v in spec_b4.items() if b4[k] != v}
print("차이:", diffs if diffs else "없음 — 완전 일치 ✓")

b5 = recs[("1200", "B5")]
print(f"\n1200!B5 cached_value = {b5['cached_value']!r}",
      "✓ ISO 8601" if b5["cached_value"] == "00:00:00" else "✗")

우리 방출  : {"sheet": "1200", "cell": "B4", "row": 4, "col": 2, "value": null, "formula": "'1100'!B4", "cached_value": 0, "data_type": "f", "number_format": "General", "merged_range": null, "bold": false, "border": true, "fill": null}
지시서 예시: {"sheet": "1200", "cell": "B4", "row": 4, "col": 2, "value": null, "formula": "'1100'!B4", "cached_value": 0, "data_type": "f", "number_format": "General", "merged_range": null, "bold": false, "border": true, "fill": null}
차이: 없음 — 완전 일치 ✓

1200!B5 cached_value = '00:00:00' ✓ ISO 8601


## 3. 포함 규칙 — 무엇이 살고 무엇이 빠지나

§4.4 포함 규칙: 값이나 수식이 있는 셀은 전부 + 값이 없어도
**(a) 병합 anchor (b) 테두리 (c) 배경색**이 있으면 포함.
반대로 **병합 자식 셀**과 **굵게(bold)만 있는 셀**은 빠집니다.

- "값 없는 테두리 셀"이 살아남는 이유: 조서의 **빈 입력 칸**(아직 안 채운 슬롯)도
  사실이므로 보존해야 하기 때문입니다.
- 병합 자식이 빠지는 이유: 병합된 덩어리는 왼쪽 위 한 칸(anchor)이 대표하고,
  anchor의 `merged_range`에 범위가 적히기 때문입니다.
- 병합됐는데 내용도 서식도 없는 anchor는 **null 값으로 합성**해서라도 넣습니다 —
  1300C 파일에서 실제로 볼 수 있습니다.

In [4]:
from openpyxl.utils import range_boundaries

# (1) 값·수식 없이 살아남은 셀 = 빈 입력 슬롯
empty_slots = [r for r in recs.values() if r["value"] is None and r["formula"] is None]
print(f"감사계약: 전체 {len(recs)}줄 중 빈 입력 슬롯 {len(empty_slots)}줄")
for r in empty_slots[:3]:
    why = "병합 anchor" if r["merged_range"] else ("테두리" if r["border"] else "배경색")
    print(f"  예: {r['sheet']}!{r['cell']} — 살아남은 이유: {why}")

# (2) 병합 자식 셀이 안 들어갔는지 전수 검사
leak = 0
for s in ir.sheets:
    in_jsonl = {(r["row"], r["col"]) for r in recs.values() if r["sheet"] == s.name}
    for ref in s.merged_ranges:
        c1, r1, c2, r2 = range_boundaries(ref)
        leak += sum(1 for rr in range(r1, r2 + 1) for cc in range(c1, c2 + 1)
                    if (rr, cc) != (r1, c1) and (rr, cc) in in_jsonl)
print(f"\n병합 자식 누출: {leak}건 (기대 0)")

# (3) 내용·서식 없는 병합 anchor의 합성 — 1300C에서 관찰
ir_1300c = extract_workbook(next(RAW.glob("*1300C*")))
synth = [r for r in iter_cell_records(ir_1300c)
         if (r["row"], r["col"]) not in next(s for s in ir_1300c.sheets if s.name == r["sheet"]).cells]
print(f"\n1300C 산업전문가검토조서: 합성된 병합 anchor {len(synth)}건")
print("  예:", json.dumps(synth[0], ensure_ascii=False) if synth else "-")

감사계약: 전체 208줄 중 빈 입력 슬롯 125줄
  예: 1100!B4 — 살아남은 이유: 테두리
  예: 1100!D4 — 살아남은 이유: 테두리
  예: 1100!F4 — 살아남은 이유: 테두리

병합 자식 누출: 0건 (기대 0)

1300C 산업전문가검토조서: 합성된 병합 anchor 16건
  예: {"sheet": "1300C", "cell": "B19", "row": 19, "col": 2, "value": null, "formula": null, "cached_value": null, "data_type": "n", "number_format": null, "merged_range": "B19:C19", "bold": false, "border": false, "fill": null}


## 4. used range 재계산 (승인 편차 D-01)

엑셀 파일 안에는 "이 시트는 어디까지 쓰였다"는 기록(dimension 레코드)이 있지만,
**믿을 수 없습니다**. 감사계약 1200 시트의 기록은 `A2:H43`인데 실제 내용은
35행 이하에만 있습니다. 그래서 기록 대신 **실제 내용이 있는 곳까지만** 범위를
다시 계산합니다(원점은 항상 A1). 이 방식이 v1.1 §5에 규칙으로 승인됐습니다.

In [5]:
import xml.etree.ElementTree as ET
import zipfile

# 파일 안의 dimension 레코드를 직접 읽어서 비교
NS_M = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
NS_R = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
NS_P = "http://schemas.openxmlformats.org/package/2006/relationships"
with zipfile.ZipFile(gyeyak) as zf:
    rels = {r.get("Id"): r.get("Target")
            for r in ET.fromstring(zf.read("xl/_rels/workbook.xml.rels")).iter(f"{{{NS_P}}}Relationship")}
    for el in ET.fromstring(zf.read("xl/workbook.xml")).iter(f"{{{NS_M}}}sheet"):
        target = rels[el.get(f"{{{NS_R}}}id")]
        target = target if target.startswith("xl/") else "xl/" + target
        dim = ET.fromstring(zf.read(target)).find(f"{{{NS_M}}}dimension")
        record = dim.get("ref") if dim is not None else "-"
        recalc = next(s.dimensions for s in ir.sheets if s.name == el.get("name"))
        mark = "  ← D-01 사례" if el.get("name") == "1200" else ""
        print(f"시트 {el.get('name'):>5}: 파일 기록 {record:>8} → 재계산 {recalc}{mark}")

시트  1100: 파일 기록   A2:F13 → 재계산 A1:F13
시트  1200: 파일 기록   A2:H43 → 재계산 A1:F35  ← D-01 사례
시트  1300: 파일 기록   A2:F20 → 재계산 A1:F16


## 5. 결정론 (P2) — 두 번 만들어도 같은가

같은 파일을 **추출부터 방출까지 두 번** 반복해서 결과 파일의 sha256 해시를
비교합니다. 한 글자라도 다르면 해시가 달라집니다. 로더 경로가 다른
대표 3종(일반 / read_only 폴백 / xls)으로 확인합니다.
이것이 verify 명령의 V3(재현성) 검사가 하게 될 일입니다.

In [6]:
samples = ["*1100~1300*", "*1300A*", "*7540*"]  # normal / read_only / xls
for pat in samples:
    p = next(RAW.glob(pat))
    hashes = []
    for run in (1, 2):
        out = OUT / f"determinism_run{run}.jsonl"
        write_cells_jsonl(extract_workbook(p), out)
        hashes.append(hashlib.sha256(out.read_bytes()).hexdigest())
        out.unlink()
    ok = "✓ 동일" if hashes[0] == hashes[1] else "✗ 불일치!"
    print(f"{ok}  {p.name[:44]}  {hashes[0][:16]}…")

✓ 동일  감사조서서식_1100~1300 감사계약.xlsx  4b7b5edbeb61337d…
✓ 동일  감사조서서식_1300A_독립성준수검토조서 2025.xlsx  c7c578c710cd7cea…
✓ 동일  감사조서서식_7540 연결공시사항점검표 (KIFRS용)_2025.xls  cdfbeb94618d4932…


## 6. 32종 전수 방출

코퍼스 전체를 방출해서 실패가 없는지, 파일별 줄 수를 봅니다.
모든 산출물은 `tests/_output/`에 남으니 직접 열어봐도 됩니다.

In [7]:
fails, total = [], 0
print(f"{'파일':<46} {'loader_path':<30} {'줄수':>7}")
for p in sorted(RAW.glob("*.xls*")):
    try:
        ir_p = extract_workbook(p)
        cnt = write_cells_jsonl(ir_p, OUT / (p.stem[:40] + ".cells.jsonl"))
        total += cnt
        print(f"{p.name[:44]:<46} {ir_p.loader_path:<30} {cnt:>7}")
    except Exception as e:
        fails.append((p.name, repr(e)))
        print(f"{p.name[:44]:<46} {'실패':<30} {'-':>7}")
print(f"\n합계 {total}줄, 실패 {len(fails)}건")
for name, err in fails:
    print(f"  ✗ {name}: {err}")

파일                                             loader_path                         줄수
감사조서서식_01 조서파일 KIFRS.xlsx                      openpyxl_normal                    267
감사조서서식_02 영구조서목록.xlsx                          openpyxl_normal                    128
감사조서서식_03 일반조서목록 KIFRS.xlsx                    openpyxl_normal                    288
감사조서서식_04 감사조서철 작성 및 보존.xlsx                   openpyxl_normal                      4
감사조서서식_1100A_계약전 위험평가표 수임 2025.xlsx            openpyxl_normal                   2549
감사조서서식_1100~1300 감사계약.xlsx                     openpyxl_normal                    208
감사조서서식_1101A_계약전 위험평가표 계속 2025.xlsx            openpyxl_normal                   3612
감사조서서식_1300A_독립성준수검토조서 2025.xlsx               openpyxl_read_only+xml_merge       237
감사조서서식_1300B_개별 감사 참여자의 독립성 준수 확인서 2025.xlsx   openpyxl_normal                    111
감사조서서식_1300C_산업전문가검토조서.xlsx                    openpyxl_normal                     37
감사조서서식_2100-2700 위험평가 2025.xlsx                openpyx